<center><h1><b>SEARCH FOR ALL PAIRS</b></h1></center>

In this notebook we want to generate all possible pairs strarting from raw tracks data

The codes in the pdg are:

$$
\begin{align}
D^0 \rightarrow K^- \quad \pi^+ \\
421 \rightarrow -321 \quad211
\end{align}
$$


$$
\begin{align}
\overline{D^0} \rightarrow& \quad K^+ \qquad \pi^- \\
-421 \rightarrow& +321 \quad-211
\end{align}
$$

In [1]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import Image, display

import gc
import itertools

---

## CHOOSE THE CHUNK TO ANALYZE

#### OUR DATA

In [2]:
%%bash
ls "../../mnt/SingleTrackTrees/Simulation/alice_sim_2024_LHC24h1_1_536237_AOD"

001_010
011_020
021_030
031_040
041_050
051_060
061_070
071_080
081_090
091_100


#### CHOOSE CHUNK

In [3]:
base_path = "../../mnt/SingleTrackTrees/Simulation/alice_sim_2024_LHC24h1_1_536237_AOD"
# which_chunk = ["001_010", "011_020", "021_030", "031_040", "041_050", "051_060", "061_070", "071_080", "081_090", "091_100" ]
which_chunk = ["061_070"]
which_file = "AO2Dtree.root"

In [4]:
# already computed pairs:
! ls /mnt/FilterResults/pairs_with_all_variabs

mc_pairs_001_010_pt1.pkl   mc_pairs_011_020_pt9.pkl   mc_pairs_021_030_pt8.pkl
mc_pairs_001_010_pt2.pkl   mc_pairs_021_030_pt1.pkl   mc_pairs_021_030_pt9.pkl
mc_pairs_001_010_pt3.pkl   mc_pairs_021_030_pt10.pkl  mc_pairs_031_040_pt1.pkl
mc_pairs_001_010_pt4.pkl   mc_pairs_021_030_pt11.pkl  mc_pairs_031_040_pt2.pkl
mc_pairs_001_010_pt5.pkl   mc_pairs_021_030_pt12.pkl  mc_pairs_031_040_pt3.pkl
mc_pairs_011_020_pt1.pkl   mc_pairs_021_030_pt13.pkl  mc_pairs_031_040_pt4.pkl
mc_pairs_011_020_pt10.pkl  mc_pairs_021_030_pt14.pkl  mc_pairs_031_040_pt5.pkl
mc_pairs_011_020_pt11.pkl  mc_pairs_021_030_pt15.pkl  mc_pairs_041_050_pt1.pkl
mc_pairs_011_020_pt12.pkl  mc_pairs_021_030_pt16.pkl  mc_pairs_041_050_pt2.pkl
mc_pairs_011_020_pt13.pkl  mc_pairs_021_030_pt17.pkl  mc_pairs_041_050_pt3.pkl
mc_pairs_011_020_pt14.pkl  mc_pairs_021_030_pt18.pkl  mc_pairs_041_050_pt4.pkl
mc_pairs_011_020_pt15.pkl  mc_pairs_021_030_pt19.pkl  mc_pairs_071_080_pt1.pkl
mc_pairs_011_020_pt2.pkl   mc_pairs_021_030_pt2.pkl 

### TEST

In [5]:
# debug
path_test = f"../../mnt/SingleTrackTrees/Simulation/alice_sim_2024_LHC24h1_1_536237_AOD/{which_chunk[0]}/AO2Dtree.root"
file_test = uproot.open(path_test)
diction = file_test.classnames()
for k, v in list(diction.items())[:7]:
    print(k, v)

DF_2303121157929504;1 TDirectory
DF_2303121157929504/O2mccollision;1 TTree
DF_2303121157929504/O2collision_001;1 TTree
DF_2303121157929504/O2filtertrack;1 TTree
DF_2303121157929504/O2filtertrackextr;1 TTree
DF_2303121157929504/O2filtertrackmc;1 TTree
DF_2303121157929504/O2genparticles;1 TTree


In [6]:
# # debug
# file_test["DF_2303121152302944/O2filtertrackmc"].show()

In [7]:
# # debug
# df_test = file_test["DF_2303121152302944/O2filtertrackmc"].arrays(library="pd")
# df_test["fMainHfMotherPdgCode"].value_counts()

As we can see, there are both $D_0$ and $anti-D_0$.

In [8]:
# debug
names_dirs = file_test.keys(filter_classname="TDirectory")
print(f"We have --{len(names_dirs)}-- directories of data in chunk --{which_chunk[0]}--.")

We have --389-- directories of data in chunk --061_070--.


### SECONDARY VERTEX FUNCTION

In [9]:
def to_global_coord( row ):
    xy_in = np.array([row["fX"],row["fY"]])
    # angle = -row["fAlpha"]
    # rot = np.array([[np.cos(angle),np.sin(angle)], [-np.sin(angle), np.cos(angle)]]) # inverse matrix of the one from to_track_coord (-alfa)
    # Rewriting the above matrix using cosine/sin even/odd function properties: less multiplications
    angle = row["fAlpha"]
    rot = np.array([[np.cos(angle),-np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    xy_out = rot.dot(xy_in)
    return( xy_out )

def secondary_vertex ( row1, row2 ):
    XY1, XY2 = [to_global_coord(row1), to_global_coord(row2)]
    x1 = XY1[0]
    y1 = XY1[1]
    x2 = XY2[0]
    y2 = XY2[1]
    
    px1 = row1["px"]
    px2 = row2["px"]
    py1 = row1["py"]
    py2 = row2["py"]
    pz1 = row1["pz"]
    pz2 = row2["pz"]
    
    m1 = py1/px1
    m2 = py2/px2
    q1 = y1 - m1*x1
    q2 = y2 - m2*x2
    x_SV = ( q2 - q1 )/( m1 - m2)
    y_SV = y1 + m1*(x_SV - x1)
   
    z1_track = pz1/px1 * (x_SV - x1) + row1["fZ"]
    z2_track = pz2/px2 * (x_SV - x2) + row2["fZ"]
    z_SV = (z1_track + z2_track)/2
    return ([x_SV, y_SV, z_SV])

# Particle masses in GeV
m_K = 0.493677
m_pi = 0.139570
m_d0 = 1.86484

## CUT

In [10]:
# cut_expression = (
#     "( (fNsigmaTOFpi > -5) & (fNsigmaTOFpi < 5) ) | "
#     "( (fNsigmaTOFka > -5) & (fNsigmaTOFka < 5) )   "
#     )

---

# PAIRS SEARCH

In [11]:
chunk = which_chunk[0]   # simple renaming
num_sections = 5         # number of slices to divide a single chunk
naming_offset= 0         # might be necessary for slicing even more the computation

path = base_path + "/".join(["/", chunk, which_file])
file = uproot.open(path)




names_track_extr = file.keys(filter_name=r"*O2filtertrackextr")
names_track      = file.keys(filter_name=r"*O2filtertrack")
names_coll       = file.keys(filter_name=r"*O2collision_001")

names_mc_gen     = file.keys(filter_name=r"*O2genparticles")
names_mc_track   = file.keys(filter_name=r"*O2filtertrackmc")
names_mc_coll    = file.keys(filter_name=r"*O2mccollision")

# LET'S SPLIT THE CHUNK IN MORE PARTS (RAM ISSUE)
tot = np.arange(0, len(names_coll) )         # indexes from 0 to max (i.e. len(names_coll) )
all_divisions = np.array_split(tot, num_sections)

for i_div, div in enumerate(all_divisions):         # cycle over n parts of the chunk
    list_of_df = []               # list with all the dataframes/directories of a single part of a chunk (we will concat them later)  
    collision_offset = 0          # variable to correct the index of the collision (because it starts from 0 at each new TTree)

    for i in div:              # cycle over all the dataframes/directories of a single part of a chunk

        # READING DIRECTORIES:
        df_coll = file[ names_coll[i] ].arrays(["fPosX", "fPosY", "fPosZ"], library="pd")
        df_trackextr = file[ names_track_extr[i] ].arrays(["fPt", "fEta", "fCharge", "fDcaXY",
             "fNsigmaTPCpi", "fNsigmaTPCka", "fNsigmaTOFpi", "fNsigmaTOFka"], library="pd" )
        df_track = file[ names_track[i] ].arrays(["fIndexCollisions", "fAlpha", "fX", "fY", "fZ"], library="pd")
        # take the corresponding data of MC and merge it in df_track
        df_track_mc = file[names_mc_track[i]].arrays(["fPdgCode","fMainMotherOrigIndex","fMainHfMotherPdgCode","fMainMotherNfinalStateDaught",
                                                      "fMainBeautyAncestorPdgCode"],library="pd")
        df_track = pd.merge(left=df_track, 
                             right=df_track_mc, 
                             how='inner', left_index=True, right_index=True)   # I simply create a new dataframe as those two side by side
        # merge track and trackextr:
        df_trackextr = pd.merge(left=df_trackextr, right=df_track, how='inner', left_index=True, right_index=True)


        # MONTE CARLO FILTERS________________________________________________________________________________________
        # # FILTER USING KNOWN TRUTH: keep all particles that satisfy both a and b:
        #     # a. come from D0 AND are either K- or pi+, 
        #     #     OR viceversa come from anti-D0 AND are either K+ or pi-
        #     # b. have fMainBeautyAncestor = 0
        # df_trackextr = df_trackextr[
        #     ( ( (df_trackextr["fMainHfMotherPdgCode"]  == 421) & (df_trackextr["fPdgCode"].isin([-321,211])) ) | # or
        #       ( (df_trackextr["fMainHfMotherPdgCode"]  ==-421) & (df_trackextr["fPdgCode"].isin([321,-211])) )  )
        #     &
        #     (df_trackextr["fMainBeautyAncestorPdgCode"]==0)
        #     ]
        # # keep only those with 2 or more daughter
        # df_trackextr = df_trackextr[ df_trackextr["fMainMotherNfinalStateDaught"] >= 2 ]

        
        # CUTS_________________________________________________________________________________________________________
        # we cut rows where the fIndexCollision is (for some unknown reason) negative 
        valid = df_trackextr["fIndexCollisions"] >= 0
        df_trackextr = df_trackextr[valid].reset_index(drop=True)  
        # # CUT ON TOF:
        # df_trackextr = df_trackextr.query(cut_expression)

        
        # KEPT VARIABLES ________________________________________________________________________________________________
        # We keep the variables over which we did the cut to see how they behave for the signal
        df_trackextr = df_trackextr[
                        ["fIndexCollisions", "fPdgCode", "fMainHfMotherPdgCode", "fPt", "fEta", "fCharge", "fDcaXY",
                          "fAlpha", "fX", "fY", "fZ", "fNsigmaTPCpi", "fNsigmaTPCka","fNsigmaTOFpi", "fNsigmaTOFka",
                          "fMainMotherOrigIndex", "fMainMotherNfinalStateDaught", "fMainBeautyAncestorPdgCode"] ]
    

        # MERGING COLLISION VARIABLES THROUGH INDEX COLLISION______________________________________________________________
        # Now we add fPosX,Y,Z as new columns (connecting them through fIndexCollisions) and cut over fPosZ
        df_trackextr["fPosZ"] = df_coll.iloc[df_trackextr["fIndexCollisions"].values]["fPosZ"].values
        df_trackextr["fPosX"] = df_coll.iloc[df_trackextr["fIndexCollisions"].values]["fPosX"].values
        df_trackextr["fPosY"] = df_coll.iloc[df_trackextr["fIndexCollisions"].values]["fPosY"].values
        df_trackextr = df_trackextr[(df_trackextr["fPosZ"] < 10) & (df_trackextr["fPosZ"] > -10)].reset_index(drop=True)
        df_trackextr["fIndexCollisions"] += collision_offset 
    
        # save results:
        # ALTERNATIVE 1: for the first cycle, let's copy the first dataframe, then we concatenate the next ones
        # if  i==0: df = df_trackextr
        # else: df = pd.concat([df, df_trackextr], ignore_index=True)
        # # ALTERNATIVE 2:
        list_of_df.append( df_trackextr )              # add the final dataframe in a list (we will concat them later)
        
        # UPDATE OFFSET for next loop:
        collision_offset += len(df_coll)

        # CLEANING SPACE
        del df_track
        del df_trackextr
        del df_coll
        del df_track_mc

        # END OF UPLOADING DATA FROM A PART OF A CHUNK_______________________________________________________________________________

    # MERGE all the dataframes/directories (of a part of a chunk) in a single dataframe
    df = pd.concat(list_of_df, ignore_index=True)
    N = len(df)
    del list_of_df

    # Debug
    print(f"Chunk {chunk}, slice {i_div+1}") 
    print(f"    The uploaded dataframe has {len(df)} rows and {len(df.columns)} columns.")
    memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"    It occupies {memory:.2f} MB")
    
    # NEW CALCULATED VARIABLES:
    # linear moments of tracks and add them as column to the dataframe:
    df["px"] = df["fPt"] * np.cos(df["fAlpha"])
    df["py"] = df["fPt"] * np.sin(df["fAlpha"])
    df["pz"] = df["fPt"] * np.sinh(df["fEta"])
    # let's add ENERGY columns (differentiating pions and kaons) in the two cases (D0 and anti-D0)
    mass = np.where(df["fCharge"] > 0, m_pi, m_K)
    anti_mass = np.where(df["fCharge"] > 0, m_K, m_pi)
    df["Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + mass**2)
    df["Anti-Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + anti_mass**2)

    # let's initialize some LISTS, then we will create a dataframe in the end
    collision_indices, track1_indices, track2_indices, dcaXY_products, inv_masses, inv_masses_approx, anti_masses = [] = ([] for _ in range(7))
    pt_totals , pz_totals, SV_X, SV_Y, SV_Z = ([] for _ in range(5))
    decay_lengths , cos_pointings =           ([] for _ in range(2))
    # filter variables behaviour
    neg_TPCka , neg_TOFka, pos_TPCpi, pos_TOFpi = ([] for _ in range(4))
    # also cross-check over the other particles's variables (e.g. the TOFpi for a ka particle)
    neg_TPCpi, neg_TOFpi, pos_TPCka, pos_TOFka =  ([] for _ in range(4))
    # also for the fDCAXY of the two daughters
    neg_fDcaXY, pos_fDcaXY =                    ([] for _ in range(2))
    # and their transverse momentum
    neg_pt, pos_pt =                            ([] for _ in range(2))
    # MC informations
    track_pdg_pos, track_pdg_neg, mother_pdg_pos, mother_pdg_neg, beauty_pdg_pos, beauty_pdg_neg = ([] for _ in range(6))
    mother_N_final_state_pos, mother_N_final_state_neg, mother_orig_ind_pos, mother_orig_ind_neg = ([] for _ in range(4))

    # counting=0 # debug variable

    # PAIRS SEARCH ______________________________________________________________________________________________________
    
    # let's divide the dataframe in positive and negative charged (only for computation convenience, not physical meaning)
    df_pos = df[ df['fCharge']>0 ]
    df_neg = df[ df['fCharge']<0 ]
    
    # Iterate over each collision group
    for collision_idx in (df_neg['fIndexCollisions'].unique()): # we cycle starting by the negative ones only because they are less than the positives
        group_pos = df_pos[ df_pos['fIndexCollisions'] == collision_idx ]
        group_neg = df_neg[ df_neg['fIndexCollisions'] == collision_idx ]
    
        # Filter for only collisions with at least a pair
        if len(group_pos) < 1:   continue
    
        # Reset index of the group to 0..N-1 and move original index in new column 'orig_index'
        group_pos = group_pos.reset_index().rename(columns={'index': 'orig_index'})
        group_neg = group_neg.reset_index().rename(columns={'index': 'orig_index'})
    
        # let's create indexes for all possible pairs:
        combinat = itertools.product( range(len(group_neg)), range(len(group_pos)) )
    
        # Iterate over all unique pairs of tracks
        for combo in combinat:
            row_neg = group_neg.iloc[combo[0]]
            row_pos = group_pos.iloc[combo[1]]
    
            product_dcaXY = row_neg['fDcaXY'] * row_pos['fDcaXY']
            
            # INVARIANT MASS calculation
            pt1, pt2 = row_neg['fPt'], row_pos['fPt']
            # eta1, eta2 = row_neg['fEta'], row_pos['fEta']
            # phi1, phi2 = row_neg['fAlpha'], row_pos['fAlpha']
            # delta_eta = eta1 - eta2
            # delta_phi = phi1 - phi2
    
            # approximation formula
            # inv_mass_approx = np.sqrt(2 * pt1 * pt2 * (np.cosh(delta_eta) - np.cos(delta_phi)))
    
            # exact formula:
            E1 = row_neg['Ene']
            E2 = row_pos['Ene']
            px1 = row_neg["px"]
            py1 = row_neg["py"]
            pz1 = row_neg["pz"]
            px2 = row_pos["px"]
            py2 = row_pos["py"]
            pz2 = row_pos["pz"]
            inv_mass = np.sqrt( (E1+E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )
            anti_E1 = row_neg['Anti-Ene']
            anti_E2 = row_pos['Anti-Ene']
            anti_inv_mass = np.sqrt( (anti_E1+anti_E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )
    
            # total TRANSVERSE MOMENTUM of the D0 candidate (used later for sliced plots)
            pt_total = np.sqrt((px1 + px2)**2 + (py1 + py2)**2)
    
            # SECONDARY VERTEX
            SV_coords = np.array( secondary_vertex(row_neg, row_pos) )
            # SV_X.append(SV_coords[0])
            # SV_Y.append(SV_coords[1])
            # SV_Z.append(SV_coords[2])
    
            # DECAY LENGTH: distance between PV and SV
            PV_coords = np.array( [row_pos["fPosX"], row_pos["fPosY"], row_pos["fPosZ"]] )
            decay_lengths.append( np.linalg.norm(SV_coords - PV_coords ) )
    
            # cosine of POINTING ANGLE: the latter is the angle between the direction of the mother particle and the line connecting PV and SV
            mother_direction = [px1+px2, py1+py2, pz1+pz2]
            flight_line = SV_coords - PV_coords
            cos_pointings.append ( np.dot(mother_direction, flight_line)/(np.linalg.norm(mother_direction)*np.linalg.norm(flight_line)) )
            
            # SAVE RESULTS in previously defined lists:
            # collision_indices.append(int(row_neg['fIndexCollisions']))
            # track1_indices.append(int(row_neg['orig_index']))
            # track2_indices.append(int(row_pos['orig_index']))
            dcaXY_products.append(product_dcaXY)
            inv_masses.append(inv_mass)
            anti_masses.append(anti_inv_mass)
            # inv_masses_approx.append(inv_mass_approx)
            pt_totals.append(pt_total)
            # pz_totals.append(pz1+pz2)

            # let's save SINGLE TRACK VARIABLES:
            # variable check: for D0, kaon is minus---> Ka<-->row neg; pi<-->row pos
            neg_pt.append(row_neg['fPt'])
            neg_TPCka.append(row_neg["fNsigmaTPCka"])
            neg_TOFka.append(row_neg["fNsigmaTOFka"])
            neg_fDcaXY.append(row_neg["fDcaXY"])
            pos_pt.append(row_pos['fPt'])
            pos_TPCpi.append(row_pos["fNsigmaTPCpi"])
            pos_TOFpi.append(row_pos["fNsigmaTOFpi"])
            pos_fDcaXY.append(row_pos["fDcaXY"])
            # also cross-check over the other particles's variables (e.g. the TOFpi for a ka particle) 
            neg_TPCpi.append(row_neg["fNsigmaTPCpi"])
            neg_TOFpi.append(row_neg["fNsigmaTOFpi"])
            pos_TPCka.append(row_pos["fNsigmaTPCka"])
            pos_TOFka.append(row_pos["fNsigmaTOFka"])

            # MC info:
            track_pdg_pos.append(row_pos["fPdgCode"])
            track_pdg_neg.append(row_neg["fPdgCode"])
            mother_pdg_pos.append(row_pos["fMainHfMotherPdgCode"])
            mother_pdg_neg.append(row_neg["fMainHfMotherPdgCode"])
            beauty_pdg_pos.append(row_pos["fMainBeautyAncestorPdgCode"])
            beauty_pdg_neg.append(row_neg["fMainBeautyAncestorPdgCode"])
            mother_N_final_state_pos.append(row_pos["fMainMotherNfinalStateDaught"])
            mother_N_final_state_neg.append(row_neg["fMainMotherNfinalStateDaught"])
            mother_orig_ind_pos.append(row_pos["fMainMotherOrigIndex"])
            mother_orig_ind_neg.append(row_neg["fMainMotherOrigIndex"])
    
        # # let's free the memory RAM of unused dataframes:
        # # PROBLEM: THIS IS VERY SLOW!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        # del group
        # gc.collect()
    
        # # debug:
        # counting += 1
        # if counting % 50000 ==0: print(counting, end=' ')
    
    # create a dataframe with the results:
    df_pairs = pd.DataFrame({
        # 'collision_index': collision_indices,
        # 'track1_index': track1_indices,
        # 'track2_index': track2_indices,
        'dcaXY_product': dcaXY_products,
        'inv_mass': inv_masses,
        'anti_mass': anti_masses,
        'pt': pt_totals,
        # 'pz': pz_totals,
        'decay_length': decay_lengths,
        'cos_pointing': cos_pointings,
        # 'SV_X': SV_X,
        # 'SV_Y': SV_Y,
        # 'SV_Z': SV_Z,
        'neg_pt': neg_pt,
        'neg_TPCka': neg_TPCka,
        'neg_TOFka': neg_TOFka,
        'neg_fDcaXY': neg_fDcaXY,
        'pos_pt': pos_pt,
        'pos_TPCpi': pos_TPCpi,
        'pos_TOFpi': pos_TOFpi,
        'pos_fDcaXY': pos_fDcaXY,
        # cross-check:
        'neg_TPCpi': neg_TPCpi,
        'neg_TOFpi': neg_TOFpi,
        'pos_TPCka': pos_TPCka,
        'pos_TOFka': pos_TOFka,
        # MC info
        'track_pdg_pos': track_pdg_pos,
        'track_pdg_neg': track_pdg_neg,
        'mother_pdg_pos': mother_pdg_pos,
        'mother_pdg_neg': mother_pdg_neg,
        'beauty_pdg_pos': beauty_pdg_pos,
        'beauty_pdg_neg': beauty_pdg_neg,
        'mother_N_final_state_pos': mother_N_final_state_pos,
        'mother_N_final_state_neg': mother_N_final_state_neg,
        'mother_orig_ind_pos': mother_orig_ind_pos,
        'mother_orig_ind_neg': mother_orig_ind_neg,
    })
    # Debug
    print(f"Chunk {chunk}, slice {i_div+1}") 
    print(f"    The computed pairs dataframe has {len(df_pairs)} rows and {len(df_pairs.columns)} columns.")
    memory = df_pairs.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"    It occupies {memory:.2f} MB")
    print(f"    Here are some rows of it:")
    # output example:
    display( pd.concat([df_pairs.head(3),df_pairs.tail(3)]) )

    # LET'S SAVE RESULTS:
    df_pairs.to_pickle(f"/mnt/FilterResults/pairs_with_all_variabs/mc_pairs_{chunk}_pt{i_div+1 + naming_offset}.pkl")

    del df_pairs
    gc.collect()

Chunk 061_070, slice 1
    The uploaded dataframe has 363666 rows and 21 columns.
    It occupies 29.13 MB
Chunk 061_070, slice 1
    The computed pairs dataframe has 1330565 rows and 28 columns.
    It occupies 284.24 MB
    Here are some rows of it:


,dcaXY_product,inv_mass,anti_mass,pt,decay_length,cos_pointing,neg_pt,neg_TPCka,neg_TOFka,neg_fDcaXY,...,track_pdg_pos,track_pdg_neg,mother_pdg_pos,mother_pdg_neg,beauty_pdg_pos,beauty_pdg_neg,mother_N_final_state_pos,mother_N_final_state_neg,mother_orig_ind_pos,mother_orig_ind_neg
0,-4.758999e-06,3.017030,2.920470,1.458452,0.003997,0.478802,0.673003,-3.287788,-24.917126,-0.001826,...,211.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1,-1.369651e-05,0.922572,1.021690,1.048824,0.011905,0.962209,0.673003,-3.287788,-24.917126,-0.001826,...,211.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
2,-6.404469e-05,1.673165,1.640110,1.186721,3.318282,-0.140620,0.673003,-3.287788,-24.917126,-0.001826,...,211.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1330562,2.690089e-07,3.498473,3.172010,6.431845,0.008004,-0.998537,0.546377,17.136892,-999.000000,0.004129,...,321.0,-2212.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1330563,-1.515839e-06,0.920596,0.825068,0.995011,0.005761,-0.545948,0.415962,-7.480197,-38.135254,0.004315,...,211.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1330564,-3.234064e-06,2.565610,2.639292,1.185176,0.018662,-0.544026,1.514955,-0.230740,-999.000000,0.009205,...,211.0,-321.0,0.0,421.0,0.0,-521.0,0.0,2.0,-1.0,1375798.0


Chunk 061_070, slice 2
    The uploaded dataframe has 452235 rows and 21 columns.
    It occupies 36.23 MB
Chunk 061_070, slice 2
    The computed pairs dataframe has 1506855 rows and 28 columns.
    It occupies 321.90 MB
    Here are some rows of it:


,dcaXY_product,inv_mass,anti_mass,pt,decay_length,cos_pointing,neg_pt,neg_TPCka,neg_TOFka,neg_fDcaXY,...,track_pdg_pos,track_pdg_neg,mother_pdg_pos,mother_pdg_neg,beauty_pdg_pos,beauty_pdg_neg,mother_N_final_state_pos,mother_N_final_state_neg,mother_orig_ind_pos,mother_orig_ind_neg
0,6.910952e-07,2.168106,2.179225,1.015773,0.002343,0.019631,1.211844,5.881763,31.794510,0.000917,...,211.0,-2212.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1,-6.172542e-06,0.747317,0.997233,1.569373,0.014725,0.974671,1.211844,5.881763,31.794510,0.000917,...,211.0,-2212.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
2,-5.459383e-07,3.348044,3.306756,0.883022,0.010278,0.932527,1.211844,5.881763,31.794510,0.000917,...,211.0,-2212.0,421.0,0.0,0.0,0.0,2.0,0.0,1239040.0,-1.0
1506852,5.450894e-05,1.515860,1.556446,0.319034,0.039583,0.258837,0.788741,1.575225,1.426412,0.006904,...,321.0,-321.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1506853,1.999525e-05,1.214176,1.173943,0.463944,0.012852,0.110983,0.460977,-6.539192,-35.132771,0.002533,...,321.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1506854,-5.159616e-05,0.866561,0.743010,0.856539,0.020925,-0.979050,0.324179,-9.160164,-999.000000,-0.006535,...,321.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0


Chunk 061_070, slice 3
    The uploaded dataframe has 457410 rows and 21 columns.
    It occupies 36.64 MB
Chunk 061_070, slice 3
    The computed pairs dataframe has 1532939 rows and 28 columns.
    It occupies 327.47 MB
    Here are some rows of it:


,dcaXY_product,inv_mass,anti_mass,pt,decay_length,cos_pointing,neg_pt,neg_TPCka,neg_TOFka,neg_fDcaXY,...,track_pdg_pos,track_pdg_neg,mother_pdg_pos,mother_pdg_neg,beauty_pdg_pos,beauty_pdg_neg,mother_N_final_state_pos,mother_N_final_state_neg,mother_orig_ind_pos,mother_orig_ind_neg
0,0.000129,1.599913,1.555533,0.539288,0.043438,0.077356,0.672437,-3.020375,207.499954,0.005261,...,321.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1,0.000001,0.923637,0.743290,2.126875,0.022551,-0.996443,0.672437,-3.020375,207.499954,0.005261,...,321.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
2,0.000129,1.461173,1.387114,0.933474,0.026232,0.334951,0.601551,-4.166760,-999.000000,0.005269,...,321.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1532936,0.000138,1.160519,1.103801,0.854550,0.015235,-0.272673,0.381550,0.566750,-999.000000,0.012710,...,2212.0,-321.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1532937,0.000011,1.175169,1.151186,0.579922,0.012793,0.701389,0.381550,0.566750,-999.000000,0.012710,...,211.0,-321.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1532938,-0.000017,1.135480,1.121108,0.318626,0.020138,-0.209720,0.381550,0.566750,-999.000000,0.012710,...,211.0,-321.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0


Chunk 061_070, slice 4
    The uploaded dataframe has 453871 rows and 21 columns.
    It occupies 36.36 MB
Chunk 061_070, slice 4
    The computed pairs dataframe has 1526549 rows and 28 columns.
    It occupies 326.11 MB
    Here are some rows of it:


,dcaXY_product,inv_mass,anti_mass,pt,decay_length,cos_pointing,neg_pt,neg_TPCka,neg_TOFka,neg_fDcaXY,...,track_pdg_pos,track_pdg_neg,mother_pdg_pos,mother_pdg_neg,beauty_pdg_pos,beauty_pdg_neg,mother_N_final_state_pos,mother_N_final_state_neg,mother_orig_ind_pos,mother_orig_ind_neg
0,2.560887e-04,0.711534,0.784437,0.762951,0.023142,0.061609,0.485081,-6.693298,-37.755455,-0.017165,...,211.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1,-7.960041e-06,0.671312,0.749624,0.837131,0.204579,-0.999804,0.506204,-4.773671,-32.966705,0.001244,...,211.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
2,1.234695e-07,0.923194,0.820860,1.486227,0.008363,-0.858978,0.506204,-4.773671,-32.966705,0.001244,...,321.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1526546,-3.796044e-06,1.604352,1.425202,0.760136,0.036949,-0.847597,0.370188,-9.528838,-999.000000,-0.012585,...,211.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1526547,2.539095e-05,1.019556,0.713877,1.432262,0.033092,0.948860,0.370188,-9.528838,-999.000000,-0.012585,...,211.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1526548,-5.797063e-06,0.997669,1.013939,0.095935,0.062896,0.401424,0.370188,-9.528838,-999.000000,-0.012585,...,211.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0


Chunk 061_070, slice 5
    The uploaded dataframe has 450148 rows and 21 columns.
    It occupies 36.06 MB
Chunk 061_070, slice 5
    The computed pairs dataframe has 1507841 rows and 28 columns.
    It occupies 322.11 MB
    Here are some rows of it:


,dcaXY_product,inv_mass,anti_mass,pt,decay_length,cos_pointing,neg_pt,neg_TPCka,neg_TOFka,neg_fDcaXY,...,track_pdg_pos,track_pdg_neg,mother_pdg_pos,mother_pdg_neg,beauty_pdg_pos,beauty_pdg_neg,mother_N_final_state_pos,mother_N_final_state_neg,mother_orig_ind_pos,mother_orig_ind_neg
0,0.000035,1.691023,1.832859,1.030913,2.649441,0.195024,1.367521,3.121024,25.406723,0.003303,...,211.0,-2212.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1,0.000002,0.959444,1.187162,1.733206,0.023079,-0.328515,1.367521,3.121024,25.406723,0.003303,...,211.0,-2212.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
2,0.000001,1.216208,1.312493,1.899090,0.004717,-0.725029,1.367521,3.121024,25.406723,0.003303,...,211.0,-2212.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1507838,0.000002,1.914122,1.829510,2.276204,0.014692,0.641744,0.719256,-1.018268,-17.687225,0.001844,...,211.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1507839,0.000030,0.748680,0.803920,1.271800,0.043969,0.980833,0.719256,-1.018268,-17.687225,0.001844,...,2212.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0
1507840,0.000007,1.457418,1.406224,1.259427,0.006299,0.481485,0.719256,-1.018268,-17.687225,0.001844,...,2212.0,-211.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.0,-1.0


In [12]:
# print(f"The final dataframe over all chunks has {len(final_results)} rows and {len(final_results.columns)} columns.")
# memory = final_results.memory_usage(deep=True).sum() / (1024 ** 2)
# print(f"It occupies {memory:.2f} MB")
# final_results

---